## Fase 4H: Análise Exploratória dos Dados de Produtividade (Parte 2 - Análise dos Dados)

Neste notebook, vamos realizar a segunda parte da análise exploratória dos dados históricos de produtividade agrícola coletados para as regiões de Nova Friburgo e Teresópolis, no estado do Rio de Janeiro. Vamos focar na análise dos dados, identificando padrões e tendências.

In [ ]:
# Configuração do ambiente
import setup_notebook
setup_notebook.setup_environment()

%matplotlib inline

## Introdução

Na primeira parte da análise exploratória (Fase 4G), realizamos o carregamento e a preparação dos dados de produtividade agrícola. Nesta segunda parte, vamos analisar esses dados para identificar padrões e tendências que possam ser úteis para o desenvolvimento de um modelo de previsão de produtividade.

Vamos focar nas seguintes análises:
1. Estatísticas descritivas
2. Análise da evolução da produtividade ao longo do tempo
3. Análise da produtividade por cultura
4. Análise da relação entre área plantada e produção
5. Análise da distribuição da produtividade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

# Configurar o estilo dos gráficos
plt.style.use('fivethirtyeight')
sns.set(style="whitegrid")

## 1. Carregamento dos Dados

Vamos carregar os dados combinados que foram exportados na primeira parte da análise.

In [ ]:
# Verificar se o arquivo existe
if os.path.exists('../../assets/dados_produtividade_combinados.csv'):
    # Carregar os dados combinados
    df_combined = pd.read_csv('../../assets/dados_produtividade_combinados.csv')
    
    print("Dados carregados com sucesso!")
    
    # Exibir informações sobre o dataframe
    print("\nInformações sobre o dataframe:")
    print(f"Número de registros: {df_combined.shape[0]}")
    print(f"Número de colunas: {df_combined.shape[1]}")
    print(f"Período: {df_combined['Ano'].min()} a {df_combined['Ano'].max()}")
    print(f"Municípios: {', '.join(df_combined['Município'].unique())}")
    print(f"Culturas: {', '.join(df_combined['Cultura'].unique())}")
    print(f"Tipos de Lavoura: {', '.join(df_combined['Tipo de Lavoura'].unique())}")
else:
    print("O arquivo de dados combinados não foi encontrado. Carregando os dados originais...")
    
    # Verificar se os arquivos originais existem
    if os.path.exists('../../assets/dados_produtividade_temporaria.csv') and os.path.exists('../../assets/dados_produtividade_permanente.csv'):
        # Carregar os dados de lavoura temporária
        df_temp = pd.read_csv('../../assets/dados_produtividade_temporaria.csv')
        
        # Carregar os dados de lavoura permanente
        df_perm = pd.read_csv('../../assets/dados_produtividade_permanente.csv')
        
        # Combinar os dados de lavoura temporária e permanente
        df_combined = pd.concat([df_temp, df_perm], ignore_index=True)
        
        # Adicionar uma coluna de tipo de lavoura
        df_combined['Tipo de Lavoura'] = df_combined['Cultura'].apply(lambda x: 'Temporária' if x in ['Alface', 'Couve', 'Brócolis', 'Repolho', 'Cenoura', 'Beterraba', 'Batata', 'Tomate', 'Pimentão', 'Abobrinha'] else 'Permanente')
        
        print("Dados carregados com sucesso!")
        
        # Exibir informações sobre o dataframe
        print("\nInformações sobre o dataframe:")
        print(f"Número de registros: {df_combined.shape[0]}")
        print(f"Número de colunas: {df_combined.shape[1]}")
        print(f"Período: {df_combined['Ano'].min()} a {df_combined['Ano'].max()}")
        print(f"Municípios: {', '.join(df_combined['Município'].unique())}")
        print(f"Culturas: {', '.join(df_combined['Cultura'].unique())}")
        print(f"Tipos de Lavoura: {', '.join(df_combined['Tipo de Lavoura'].unique())}")
    else:
        print("Os arquivos de dados não foram encontrados. Executando simulação...")
        
        # Simulação dos dados (código omitido por brevidade - ver notebook anterior)

## 2. Análise Exploratória dos Dados

Vamos realizar uma análise exploratória dos dados para entender suas características e identificar padrões e tendências.

### 2.1 Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas do dataframe combinado
print("Estatísticas descritivas do dataframe combinado:")
display(df_combined.describe())

# Estatísticas descritivas por tipo de lavoura
print("\nEstatísticas descritivas por tipo de lavoura:")
for tipo in df_combined['Tipo de Lavoura'].unique():
    print(f"\nTipo de Lavoura: {tipo}")
    display(df_combined[df_combined['Tipo de Lavoura'] == tipo].describe())

# Estatísticas descritivas por município
print("\nEstatísticas descritivas por município:")
for mun in df_combined['Município'].unique():
    print(f"\nMunicípio: {mun}")
    display(df_combined[df_combined['Município'] == mun].describe())

### 2.2 Análise da Evolução da Produtividade ao Longo do Tempo

In [ ]:
# Calcular a produtividade média por município, tipo de lavoura e ano
prod_by_mun_tipo_year = df_combined.groupby(['Município', 'Tipo de Lavoura', 'Ano']).agg({
    'Área Plantada (ha)': 'sum',
    'Produção (t)': 'sum'
}).reset_index()

# Calcular a produtividade média
prod_by_mun_tipo_year['Produtividade Média (t/ha)'] = prod_by_mun_tipo_year['Produção (t)'] / prod_by_mun_tipo_year['Área Plantada (ha)']

# Criar um gráfico de linhas para a evolução da produtividade média por município e tipo de lavoura
plt.figure(figsize=(14, 8))

# Definir cores e marcadores para cada combinação de município e tipo de lavoura
colors = {'Nova Friburgo': 'green', 'Teresópolis': 'blue'}
markers = {'Temporária': 'o', 'Permanente': 's'}
linestyles = {'Temporária': '-', 'Permanente': '--'}

# Plotar a evolução da produtividade média para cada combinação de município e tipo de lavoura
for mun in prod_by_mun_tipo_year['Município'].unique():
    for tipo in prod_by_mun_tipo_year['Tipo de Lavoura'].unique():
        data = prod_by_mun_tipo_year[(prod_by_mun_tipo_year['Município'] == mun) & (prod_by_mun_tipo_year['Tipo de Lavoura'] == tipo)]
        plt.plot(data['Ano'], data['Produtividade Média (t/ha)'], 
                 marker=markers[tipo], linestyle=linestyles[tipo], color=colors[mun], 
                 linewidth=2, label=f'{mun} - {tipo}')

plt.title('Evolução da Produtividade Média por Município e Tipo de Lavoura (2000-2023)')
plt.xlabel('Ano')
plt.ylabel('Produtividade Média (t/ha)')
plt.legend()
plt.grid(True)
plt.xticks(range(2000, 2024, 2))  # Mostrar apenas anos pares para melhor visualização
plt.tight_layout()
plt.show()

### 2.3 Análise da Produtividade por Cultura

In [ ]:
# Filtrar os dados para o ano de 2023
df_2023 = df_combined[df_combined['Ano'] == 2023]

# Calcular a produtividade média por cultura
prod_by_crop = df_2023.groupby(['Cultura', 'Tipo de Lavoura']).agg({
    'Área Plantada (ha)': 'sum',
    'Produção (t)': 'sum'
}).reset_index()

# Calcular a produtividade média
prod_by_crop['Produtividade Média (t/ha)'] = prod_by_crop['Produção (t)'] / prod_by_crop['Área Plantada (ha)']

# Ordenar por produtividade
prod_by_crop = prod_by_crop.sort_values('Produtividade Média (t/ha)', ascending=False)

# Criar um gráfico de barras para a produtividade média por cultura
plt.figure(figsize=(14, 8))

# Definir cores para cada tipo de lavoura
colors = {'Temporária': 'skyblue', 'Permanente': 'lightgreen'}

# Criar um gráfico de barras com cores diferentes para cada tipo de lavoura
sns.barplot(x='Cultura', y='Produtividade Média (t/ha)', data=prod_by_crop, 
            hue='Tipo de Lavoura', palette=colors)

plt.title('Produtividade Média por Cultura (2023)')
plt.xlabel('Cultura')
plt.ylabel('Produtividade Média (t/ha)')
plt.xticks(rotation=45)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

### 2.4 Análise da Relação entre Área Plantada e Produção

In [ ]:
# Criar um gráfico de dispersão para a relação entre área plantada e produção
plt.figure(figsize=(14, 8))

# Definir cores para cada tipo de lavoura
colors = {'Temporária': 'skyblue', 'Permanente': 'lightgreen'}

# Criar um gráfico de dispersão com cores diferentes para cada tipo de lavoura
for tipo in df_combined['Tipo de Lavoura'].unique():
    data = df_combined[df_combined['Tipo de Lavoura'] == tipo]
    plt.scatter(data['Área Plantada (ha)'], data['Produção (t)'], 
                alpha=0.5, label=tipo, color=colors[tipo])

# Adicionar linha de tendência para cada tipo de lavoura
for tipo in df_combined['Tipo de Lavoura'].unique():
    data = df_combined[df_combined['Tipo de Lavoura'] == tipo]
    x = data['Área Plantada (ha)']
    y = data['Produção (t)']
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    plt.plot(x, p(x), color=colors[tipo], linestyle='--')

plt.title('Relação entre Área Plantada e Produção')
plt.xlabel('Área Plantada (ha)')
plt.ylabel('Produção (t)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### 2.5 Análise da Distribuição da Produtividade

In [ ]:
# Criar um histograma para a distribuição da produtividade
plt.figure(figsize=(14, 8))

# Definir cores para cada tipo de lavoura
colors = {'Temporária': 'skyblue', 'Permanente': 'lightgreen'}

# Criar um histograma para cada tipo de lavoura
for tipo in df_combined['Tipo de Lavoura'].unique():
    data = df_combined[df_combined['Tipo de Lavoura'] == tipo]
    sns.histplot(data['Produtividade (t/ha)'], bins=30, alpha=0.5, label=tipo, color=colors[tipo])

plt.title('Distribuição da Produtividade por Tipo de Lavoura')
plt.xlabel('Produtividade (t/ha)')
plt.ylabel('Frequência')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### 2.6 Análise da Variação da Produtividade por Município e Cultura

In [ ]:
# Criar um boxplot para a variação da produtividade por município e cultura
plt.figure(figsize=(16, 10))

# Criar um boxplot para cada município
for i, mun in enumerate(df_combined['Município'].unique()):
    plt.subplot(1, 2, i+1)
    data = df_combined[df_combined['Município'] == mun]
    sns.boxplot(x='Cultura', y='Produtividade (t/ha)', data=data, hue='Tipo de Lavoura', palette=colors)
    plt.title(f'Variação da Produtividade por Cultura em {mun}')
    plt.xlabel('Cultura')
    plt.ylabel('Produtividade (t/ha)')
    plt.xticks(rotation=45)
    plt.grid(True, axis='y')
    plt.legend(title='Tipo de Lavoura')

plt.tight_layout()
plt.show()

### 2.7 Análise da Correlação entre Variáveis

In [ ]:
# Calcular a matriz de correlação
corr = df_combined[['Área Plantada (ha)', 'Produção (t)', 'Produtividade (t/ha)', 'Valor da Produção (mil R$)', 'Ano']].corr()

# Criar um mapa de calor para a matriz de correlação
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Matriz de Correlação entre Variáveis')
plt.tight_layout()
plt.show()

## 3. Avaliação do Potencial para Prever Produtividade

Com base nas análises realizadas, vamos avaliar o potencial dos dados para prever a produtividade agrícola.

### 3.1 Tendências Temporais na Produtividade

In [ ]:
# Calcular a produtividade média por ano
prod_by_year = df_combined.groupby('Ano').agg({
    'Área Plantada (ha)': 'sum',
    'Produção (t)': 'sum'
}).reset_index()

# Calcular a produtividade média
prod_by_year['Produtividade Média (t/ha)'] = prod_by_year['Produção (t)'] / prod_by_year['Área Plantada (ha)']

# Criar um gráfico de linhas para a evolução da produtividade média por ano
plt.figure(figsize=(14, 8))
plt.plot(prod_by_year['Ano'], prod_by_year['Produtividade Média (t/ha)'], marker='o', linewidth=2)

# Adicionar linha de tendência
z = np.polyfit(prod_by_year['Ano'], prod_by_year['Produtividade Média (t/ha)'], 1)
p = np.poly1d(z)
plt.plot(prod_by_year['Ano'], p(prod_by_year['Ano']), 'r--')

plt.title('Evolução da Produtividade Média ao Longo do Tempo (2000-2023)')
plt.xlabel('Ano')
plt.ylabel('Produtividade Média (t/ha)')
plt.grid(True)
plt.xticks(range(2000, 2024, 2))  # Mostrar apenas anos pares para melhor visualização
plt.tight_layout()
plt.show()

### 3.2 Relação entre Produtividade e Ano por Cultura

In [ ]:
# Selecionar algumas culturas representativas
selected_crops = ['Tomate', 'Alface', 'Batata', 'Morango', 'Maçã']

# Filtrar os dados para as culturas selecionadas
df_selected = df_combined[df_combined['Cultura'].isin(selected_crops)]

# Criar um gráfico de dispersão para a relação entre ano e produtividade por cultura
plt.figure(figsize=(14, 8))

# Definir cores para cada cultura
colors = {'Tomate': 'red', 'Alface': 'green', 'Batata': 'brown', 'Morango': 'pink', 'Maçã': 'orange'}

# Criar um gráfico de dispersão com cores diferentes para cada cultura
for crop in selected_crops:
    data = df_selected[df_selected['Cultura'] == crop]
    plt.scatter(data['Ano'], data['Produtividade (t/ha)'], 
                alpha=0.7, label=crop, color=colors[crop])
    
    # Adicionar linha de tendência para cada cultura
    z = np.polyfit(data['Ano'], data['Produtividade (t/ha)'], 1)
    p = np.poly1d(z)
    plt.plot(data['Ano'], p(data['Ano']), color=colors[crop], linestyle='--')

plt.title('Relação entre Ano e Produtividade por Cultura')
plt.xlabel('Ano')
plt.ylabel('Produtividade (t/ha)')
plt.legend()
plt.grid(True)
plt.xticks(range(2000, 2024, 2))  # Mostrar apenas anos pares para melhor visualização
plt.tight_layout()
plt.show()

## Conclusão

Neste notebook, realizamos uma análise exploratória dos dados de produtividade agrícola para as regiões de Nova Friburgo e Teresópolis. Identificamos padrões e tendências que podem ser úteis para o desenvolvimento de um modelo de previsão de produtividade.

Principais conclusões:

1. **Tendência de Crescimento**: Observamos uma tendência de crescimento na produtividade ao longo dos anos para ambos os municípios e tipos de lavoura, o que indica melhorias nas técnicas de cultivo e/ou condições favoráveis.

2. **Diferenças entre Municípios**: Nova Friburgo apresenta, em geral, produtividade ligeiramente superior à de Teresópolis, o que pode estar relacionado a fatores como solo, clima ou técnicas de cultivo.

3. **Diferenças entre Tipos de Lavoura**: A lavoura temporária apresenta produtividade média superior à lavoura permanente, o que é esperado devido às características das culturas.

4. **Variação por Cultura**: Há uma grande variação na produtividade entre as diferentes culturas, com destaque para o tomate, o pimentão e a abobrinha entre as culturas de lavoura temporária, e o morango entre as culturas de lavoura permanente.

5. **Relação Linear entre Área Plantada e Produção**: Existe uma forte relação linear entre a área plantada e a produção, o que é esperado, mas a inclinação da reta varia de acordo com o tipo de lavoura, refletindo as diferenças de produtividade.

6. **Correlação entre Variáveis**: Há uma forte correlação positiva entre produção e área plantada, e entre produção e valor da produção. A produtividade também apresenta correlação positiva com o ano, indicando uma tendência de aumento ao longo do tempo.

7. **Potencial para Previsão**: Os dados apresentam padrões claros e tendências consistentes, o que sugere um bom potencial para o desenvolvimento de um modelo de previsão de produtividade. A correlação positiva entre produtividade e ano indica que o tempo é um fator importante a ser considerado no modelo.

Na próxima fase do projeto, vamos correlacionar esses dados de produtividade com os índices vegetativos obtidos da plataforma SATVeg, buscando identificar relações que possam ser utilizadas para prever a produtividade agrícola com base nos índices vegetativos.